In [7]:
"""
rigid_lattice_diffuse.py

A first-crack implementation of rigid-body lattice-dynamics diffuse scattering,
following the same logic as your helical-spring notebook, extended to a real
crystal (PDB 6O2H, hen lysozyme, space group P1, one molecule per unit cell).

PHYSICAL MODEL
--------------
Each unit cell contains one rigid body (the protein). Every symmetry-unique
crystal contact (a lattice translation T for which some atom of the reference
molecule comes within `cutoff` of the translated copy) is given an independent,
symmetric 6x6 stiffness matrix K_bond, exactly analogous to the "Krel" you
computed for the two-cube spring toy model:

    - rows/cols 0:3 = relative translation of the two molecules' centers of mass
    - rows/cols 3:6 = relative rotation

K_bond is defined in the SAME frame for every bond: components expressed in the
crystal's fixed Cartesian (orthogonal) basis, referenced to each molecule's own
center of mass. This is the "natural" reference frame discussed in chat:
    * COM per rigid body -> no reference-point artifacts, transforms simply
      under crystallographic symmetry (identity here, since P1 has none)
    * one shared Cartesian basis for vector components -> Bloch sums over
      the lattice are simple sums of 6x6 blocks, no per-body coordinate
      changes needed.

Given K_bond and the geometric separation d = COM_2 - COM_1 = T (a known
lattice vector, since both copies have the same orientation in P1), the full
12x12 pair-Hessian is reconstructed via

    H_pair = B(d)^T @ K_bond @ B(d)

exactly as in your notebook. This construction *automatically* satisfies the
acoustic sum rule (whole-crystal rigid translation/rotation costs zero energy)
for *any* symmetric K_bond you choose -- that's the practical payoff of this
gauge: you can directly refine K_bond against the diffuse data without also
having to enforce global rigid-motion invariance by hand.

K_bond here is only a PLACEHOLDER (isotropic, scaled by the number of atomic
contacts in that bond) -- replace `guess_K_bond()` with your refined values.

DIFFUSE SCATTERING
-------------------
Standard one-phonon lattice-dynamics formula:

    D(Q)  = sum_bonds [ HAA(T) + HBB(T) ]
          + sum_bonds [ HAB(T) exp(i Q.T) + HAB(T)^T exp(-i Q.T) ]

    v(Q)  = sum_atoms f_j(Q) exp(i Q.r_j) * [ Qx,Qy,Qz, (r_j x Q)_x,(r_j x Q)_y,(r_j x Q)_z ]
            (r_j measured from the molecule's own center of mass)

    I(Q) ~ v(Q) @ D(Q)^-1 @ v(Q)^H     (real, >=0; relative units)

This is the same "structure-factor-derivative" construction used in
Meisburger, Case & Ando, Nat Commun 11:1271 (2020) -- see onePhononStructureFactors.m
in github.com/ando-lab/mdx-lib (branch `natcomm`) for the reference
implementation this was checked against.
"""

import numpy as np
import gemmi


# ---------------------------------------------------------------------------
# Small rigid-body linear algebra (same conventions as the helical-spring notebook)
# ---------------------------------------------------------------------------

def cross_matrix(r):
    return np.array([[0, -r[2], r[1]],
                      [r[2], 0, -r[0]],
                      [-r[1], r[0], 0]])


def A(r):
    """maps (u, theta) -> displacement of a point offset r from the reference point"""
    return np.block([np.eye(3), -cross_matrix(r)])


def relative_pose_matrix(d):
    """12-dim (uA,thA,uB,thB) -> 6-dim (u_rel, th_rel), d = COM_B - COM_A"""
    I = np.eye(3)
    Z = np.zeros((3, 3))
    X = cross_matrix(d)
    return np.block([[-I, X, I, Z],
                      [Z, -I, Z, I]])


def pair_hessian_from_Kbond(Kbond, d):
    """H_pair (12x12) = B(d)^T Kbond B(d); returns the 4 6x6 blocks."""
    B = relative_pose_matrix(d)
    H = B.T @ Kbond @ B
    HAA = H[0:6, 0:6]
    HAB = H[0:6, 6:12]
    HBA = H[6:12, 0:6]
    HBB = H[6:12, 6:12]
    return HAA, HAB, HBA, HBB


# ---------------------------------------------------------------------------
# Structure loading
# ---------------------------------------------------------------------------

def load_structure(pdb_path, heavy_only=True, single_altloc=True):
    """
    Returns:
        pos      (N,3) Cartesian atom positions [A]
        elements list of element symbols, length N
        occ      (N,) occupancies
        cell     gemmi.UnitCell
    """
    st = gemmi.read_structure(pdb_path)
    st.setup_entities()
    cell = st.cell
    model = st[0]

    pos, elements, occ = [], [], []
    for chain in model:
        for res in chain:
            if res.het_flag == 'H' and res.name in ('HOH',):
                continue  # skip ordered waters for the rigid-body model
            seen_altloc = set()
            for atom in res:
                if heavy_only and atom.element.name == 'H':
                    continue
                if single_altloc:
                    # keep only the first-seen alt-conformer per atom name
                    key = (res.seqid.num, atom.name)
                    if atom.altloc and atom.altloc not in ('', '\x00'):
                        if key in seen_altloc:
                            continue
                        seen_altloc.add(key)
                pos.append([atom.pos.x, atom.pos.y, atom.pos.z])
                elements.append(atom.element.name)
                occ.append(atom.occ)
    return np.array(pos), elements, np.array(occ), cell


def element_mass(sym):
    return gemmi.Element(sym).weight


def form_factor(sym, Q):
    """Q: (...,3) Cartesian scattering vectors [1/A], physics convention (F ~ exp(i Q.r)).
    Returns f(|Q|) via the IT92 analytic approximation (independent-atom model,
    no B-factor -- ADPs would multiply this by exp(-Q.U.Q/2) if you want them)."""
    el = gemmi.Element(sym)
    Qmag = np.linalg.norm(Q, axis=-1)
    stol2 = (Qmag / (4 * np.pi)) ** 2  # (sin(theta)/lambda)^2 in A^-2
    # calculate_sf is scalar-only in gemmi -- vectorize by hand
    flat = stol2.ravel()
    out = np.array([el.it92.calculate_sf(float(s)) for s in flat])
    return out.reshape(Qmag.shape)


# ---------------------------------------------------------------------------
# Crystal-contact search (periodic images), analogous to Cell.contactSearch in mdx-lib
# ---------------------------------------------------------------------------

def find_bonds(pos, cell, cutoff=4.0, nshell=2):
    """
    Find lattice translations T (n1,n2,n3 within +-nshell) for which any atom
    of the reference cell is within `cutoff` of the translated copy, and
    count contacts per bond. Returns dict {(n1,n2,n3): n_contacts} keeping
    only one representative of each +-T pair (T and -T are the same physical
    bond, viewed from either molecule).
    """
    from scipy.spatial import cKDTree

    tree = cKDTree(pos)
    lattice_vecs = np.array([
        [cell.orthogonalize(gemmi.Fractional(1, 0, 0)).x,
         cell.orthogonalize(gemmi.Fractional(1, 0, 0)).y,
         cell.orthogonalize(gemmi.Fractional(1, 0, 0)).z],
        [cell.orthogonalize(gemmi.Fractional(0, 1, 0)).x,
         cell.orthogonalize(gemmi.Fractional(0, 1, 0)).y,
         cell.orthogonalize(gemmi.Fractional(0, 1, 0)).z],
        [cell.orthogonalize(gemmi.Fractional(0, 0, 1)).x,
         cell.orthogonalize(gemmi.Fractional(0, 0, 1)).y,
         cell.orthogonalize(gemmi.Fractional(0, 0, 1)).z],
    ])

    bonds = {}
    seen = set()
    for n1 in range(-nshell, nshell + 1):
        for n2 in range(-nshell, nshell + 1):
            for n3 in range(-nshell, nshell + 1):
                if (n1, n2, n3) == (0, 0, 0):
                    continue
                if (-n1, -n2, -n3) in seen:
                    continue  # T and -T are the same bond
                T = n1 * lattice_vecs[0] + n2 * lattice_vecs[1] + n3 * lattice_vecs[2]
                shifted = pos + T
                pairs = tree.query_ball_point(shifted, r=cutoff)
                n_contacts = sum(len(p) for p in pairs)
                if n_contacts > 0:
                    bonds[(n1, n2, n3)] = {'T': T, 'n_contacts': n_contacts}
                    seen.add((n1, n2, n3))
    return bonds


# ---------------------------------------------------------------------------
# Placeholder K_bond -- REPLACE with refined values once you fit to data
# ---------------------------------------------------------------------------

def guess_K_bond(n_contacts, k_trans=0.5, k_rot=5.0):
    """
    Simple isotropic placeholder: translational and rotational stiffness
    scaled by the number of atomic contacts forming the bond. Purely
    illustrative -- the whole point of your project is to replace this with
    K matrices refined against the measured diffuse pattern.
    """
    k_t = k_trans * n_contacts
    k_r = k_rot * n_contacts
    return np.diag([k_t, k_t, k_t, k_r, k_r, k_r]).astype(float)


# ---------------------------------------------------------------------------
# Dynamical matrix and diffuse intensity
# ---------------------------------------------------------------------------

def build_bond_blocks(bonds, K_bond_func):
    """Precompute the 4 6x6 blocks for every bond."""
    blocks = {}
    for key, b in bonds.items():
        Kbond = K_bond_func(b['n_contacts'])
        HAA, HAB, HBA, HBB = pair_hessian_from_Kbond(Kbond, b['T'])
        blocks[key] = {'T': b['T'], 'HAA': HAA, 'HAB': HAB, 'HBB': HBB}
    return blocks


def dynamical_matrix(Q, blocks):
    """D(Q), a 6x6 Hermitian complex matrix, for scattering vector Q (Cartesian, 1/A,
    physics convention so that Fourier phases are exp(i Q.r))."""
    D = np.zeros((6, 6), dtype=complex)
    for b in blocks.values():
        D += b['HAA'] + b['HBB']
        phase = np.exp(1j * np.dot(Q, b['T']))
        D += b['HAB'] * phase + b['HAB'].T * np.conj(phase)
    return D


def structure_factor_derivative_vector(Q, pos_rel_com, elements):
    """v(Q), a length-6 complex row vector."""
    f = np.array([form_factor(el, Q) for el in elements])  # (N,)
    phase = np.exp(1j * (pos_rel_com @ Q))                  # (N,)
    weighted = f * phase                                     # (N,)
    r_cross_Q = np.cross(pos_rel_com, Q)                     # (N,3)
    v = np.zeros(6, dtype=complex)
    v[0:3] = np.sum(weighted[:, None] * Q[None, :], axis=0)
    v[3:6] = np.sum(weighted[:, None] * r_cross_Q, axis=0)
    return v


def diffuse_intensity(Q, blocks, pos_rel_com, elements, floor_frac=1e-6):
    """
    v(Q) @ D(Q)^-1 @ v(Q)^H via an eigendecomposition of D(Q), which is more
    numerically robust than a direct solve when D(Q) is close to singular
    (i.e. close to a soft/unstable mode of whatever K_bond you plugged in --
    this is a real physical possibility for a guessed placeholder network,
    not just a numerical nuisance: a proper refinement should check that
    D(Q) stays positive-definite over the whole Brillouin zone).
    """
    D = dynamical_matrix(Q, blocks)
    v = structure_factor_derivative_vector(Q, pos_rel_com, elements)
    w, U = np.linalg.eigh(D)  # D is Hermitian by construction: D = U diag(w) U^H
    scale = max(w.max(), 1e-12)
    w_floored = np.where(w > floor_frac * scale, w, np.inf)  # drop near-zero/negative modes
    vU = v @ U  # (vU)_j = sum_k v_k U_kj
    return float(np.sum(np.abs(vU) ** 2 / w_floored))


# ---------------------------------------------------------------------------
# Self-contained synthetic demo (no PDB needed) -- lets you confirm the whole
# pipeline (bond search -> K_bond -> dynamical matrix -> diffuse map) produces
# a sane result before pointing it at real (and much bigger) protein data.
# ---------------------------------------------------------------------------

def demo_synthetic(out_png='synthetic_diffuse_demo.png'):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    # a tiny toy "molecule": 6 identical atoms in a cross shape, packed into a
    # cubic P1 cell small enough that each arm touches its neighbor's arm.
    cell = gemmi.UnitCell(2.0, 2.0, 2.0, 90, 90, 90)
    pos = np.array([
        [0.9, 0, 0], [-0.9, 0, 0],
        [0, 0.9, 0], [0, -0.9, 0],
        [0, 0, 0.9], [0, 0, -0.9],
    ])
    elements = ['C'] * 6

    bonds = find_bonds(pos, cell, cutoff=0.35, nshell=1)
    masses = np.array([element_mass(e) for e in elements])
    com = (masses[:, None] * pos).sum(axis=0) / masses.sum()
    pos_rel_com = pos - com
    blocks = build_bond_blocks(bonds, guess_K_bond)

    D0 = dynamical_matrix(np.zeros(3), blocks)
    print('demo D(Q=0) eigenvalues:', np.round(np.linalg.eigvalsh(D0), 4),
          ' (3 exact zeros = rigid translation of the whole lattice)')

    # a 2D slice of reciprocal space, Q = (qx, qy, 0.2) in 1/A, well clear of
    # the qz=0 Bragg plane so we are looking at genuine diffuse scattering
    n = 120
    qs = np.linspace(-4, 4, n)
    I = np.zeros((n, n))
    for i, qx in enumerate(qs):
        for j, qy in enumerate(qs):
            I[j, i] = diffuse_intensity(np.array([qx, qy, 0.2]), blocks, pos_rel_com, elements)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(np.log10(I + 1e-12), origin='lower',
                    extent=[qs[0], qs[-1], qs[0], qs[-1]], cmap='inferno')
    ax.set_xlabel('$Q_x$ (1/Å)')
    ax.set_ylabel('$Q_y$ (1/Å)')
    ax.set_title('Synthetic one-phonon diffuse map, log$_{10}$ I(Q)  ($Q_z$ = 0.2 Å$^{-1}$)')
    fig.colorbar(im, ax=ax, label='log$_{10}$ I (relative units)')
    fig.tight_layout()
    fig.savefig(out_png, dpi=150)
    print(f'wrote {out_png}')

In [10]:
# ---------------------------------------------------------------------------
# Demo driver
# ---------------------------------------------------------------------------

if __name__ == '__main__':
    import sys

    if len(sys.argv) > 1 and sys.argv[1] == '--demo':
        demo_synthetic()
        sys.exit(0)

    pdb_path = sys.argv[1] if len(sys.argv) > 1 else 'pdbdata/6O2H.cif'
    cutoff = 4.0

    pos, elements, occ, cell = load_structure(pdb_path)
    print(f'{pdb_path}: {len(pos)} atoms, cell = {cell.a:.2f} {cell.b:.2f} {cell.c:.2f}  '
          f'{cell.alpha:.1f} {cell.beta:.1f} {cell.gamma:.1f}')

    masses = np.array([element_mass(e) for e in elements])
    com = (masses[:, None] * pos).sum(axis=0) / masses.sum()
    pos_rel_com = pos - com
    print(f'center of mass: {com}')

    bonds = find_bonds(pos, cell, cutoff=cutoff, nshell=2)
    print(f'found {len(bonds)} symmetry-unique crystal contacts within {cutoff} A:')
    for (n1, n2, n3), b in sorted(bonds.items()):
        print(f'  T=({n1:2d},{n2:2d},{n3:2d})  {b["n_contacts"]:4d} atom-atom contacts  '
              f'|T|={np.linalg.norm(b["T"]):.2f} A')

    if len(bonds) == 0:
        print('\nNo contacts found -- this is expected if you are running the bundled '
              'partial structure (residues 1-29 only; see note in chat). Point this '
              'script at your own complete local copy of 6o2h.pdb to get the real result.')
        sys.exit(0)

    blocks = build_bond_blocks(bonds, guess_K_bond)

    # sanity check: D(Q=0) should be positive *semi*-definite with the 6
    # acoustic zero modes near zero (rigid translation/rotation of the whole
    # crystal), same check you ran on Hr/Hl in the notebook.
    D0 = dynamical_matrix(np.zeros(3), blocks)
    eigs = np.linalg.eigvalsh(D0)
    print(f'\nD(Q=0) eigenvalues: {np.round(eigs, 4)}')
    print('Exactly 3 of these should be ~0 (uniform rigid translation of the whole '
          'crystal -- the true acoustic zero modes). The other 3 are the zone-center '
          '"libration" (rotational) mode frequencies: these are physical and generally '
          'nonzero for an infinite periodic lattice, unlike the isolated two-body case '
          'in your notebook where all 6 rigid motions of the *pair* cost zero energy.')

RuntimeError: Unknown format of -f.

In [12]:
# ---------------------------------------------------------------------------
# Demo driver -- Jupyter notebook version
# ---------------------------------------------------------------------------

# Set these manually
pdb_path = 'pdbdata/6O2H.cif'
cutoff = 4.0

# Optional: run the synthetic demo instead
run_demo = True

if run_demo:
    demo_synthetic()

else:
    pos, elements, occ, cell = load_structure(pdb_path)

    print(
        f'{pdb_path}: {len(pos)} atoms, '
        f'cell = {cell.a:.2f} {cell.b:.2f} {cell.c:.2f}  '
        f'{cell.alpha:.1f} {cell.beta:.1f} {cell.gamma:.1f}'
    )

    masses = np.array([element_mass(e) for e in elements])

    com = (masses[:, None] * pos).sum(axis=0) / masses.sum()
    pos_rel_com = pos - com

    print(f'center of mass: {com}')

    bonds = find_bonds(pos, cell, cutoff=cutoff, nshell=2)

    print(
        f'found {len(bonds)} symmetry-unique crystal contacts '
        f'within {cutoff} A:'
    )

    for (n1, n2, n3), b in sorted(bonds.items()):
        print(
            f'  T=({n1:2d},{n2:2d},{n3:2d})  '
            f'{b["n_contacts"]:4d} atom-atom contacts  '
            f'|T|={np.linalg.norm(b["T"]):.2f} A'
        )

    if len(bonds) == 0:
        print(
            '\nNo contacts found -- this is expected if you are running the bundled '
            'partial structure (residues 1-29 only; see note in chat). Point this '
            'script at your own complete local copy of 6o2h.pdb to get the real result.'
        )

    else:
        blocks = build_bond_blocks(bonds, guess_K_bond)

        # Sanity check at Gamma
        D0 = dynamical_matrix(np.zeros(3), blocks)
        eigs = np.linalg.eigvalsh(D0)

        print(f'\nD(Q=0) eigenvalues: {np.round(eigs, 4)}')

        print(
            '\nExactly 3 of these should be ~0 (uniform rigid translation of the '
            'whole crystal -- the true acoustic zero modes). The other 3 are the '
            'zone-center "libration" (rotational) mode frequencies: these are '
            'physical and generally nonzero for an infinite periodic lattice.'
        )

demo D(Q=0) eigenvalues: [0. 0. 0. 4. 4. 4.]  (3 exact zeros = rigid translation of the whole lattice)
wrote synthetic_diffuse_demo.png


In [13]:
# ---------------------------------------------------------------------------
# Demo driver -- Jupyter notebook version
# ---------------------------------------------------------------------------

# Set these manually
pdb_path = 'pdbdata/6O2H.cif'
cutoff = 4.0

# Optional: run the synthetic demo instead
run_demo = False

if run_demo:
    demo_synthetic()

else:
    pos, elements, occ, cell = load_structure(pdb_path)

    print(
        f'{pdb_path}: {len(pos)} atoms, '
        f'cell = {cell.a:.2f} {cell.b:.2f} {cell.c:.2f}  '
        f'{cell.alpha:.1f} {cell.beta:.1f} {cell.gamma:.1f}'
    )

    masses = np.array([element_mass(e) for e in elements])

    com = (masses[:, None] * pos).sum(axis=0) / masses.sum()
    pos_rel_com = pos - com

    print(f'center of mass: {com}')

    bonds = find_bonds(pos, cell, cutoff=cutoff, nshell=2)

    print(
        f'found {len(bonds)} symmetry-unique crystal contacts '
        f'within {cutoff} A:'
    )

    for (n1, n2, n3), b in sorted(bonds.items()):
        print(
            f'  T=({n1:2d},{n2:2d},{n3:2d})  '
            f'{b["n_contacts"]:4d} atom-atom contacts  '
            f'|T|={np.linalg.norm(b["T"]):.2f} A'
        )

    if len(bonds) == 0:
        print(
            '\nNo contacts found -- this is expected if you are running the bundled '
            'partial structure (residues 1-29 only; see note in chat). Point this '
            'script at your own complete local copy of 6o2h.pdb to get the real result.'
        )

    else:
        blocks = build_bond_blocks(bonds, guess_K_bond)

        # Sanity check at Gamma
        D0 = dynamical_matrix(np.zeros(3), blocks)
        eigs = np.linalg.eigvalsh(D0)

        print(f'\nD(Q=0) eigenvalues: {np.round(eigs, 4)}')

        print(
            '\nExactly 3 of these should be ~0 (uniform rigid translation of the '
            'whole crystal -- the true acoustic zero modes). The other 3 are the '
            'zone-center "libration" (rotational) mode frequencies: these are '
            'physical and generally nonzero for an infinite periodic lattice.'
        )

pdbdata/6O2H.cif: 1030 atoms, cell = 27.42 32.13 34.51  88.7 108.5 111.9
center of mass: [-1.01361052 14.49904924 24.2667288 ]
found 6 symmetry-unique crystal contacts within 4.0 A:
  T=(-1,-1, 0)    30 atom-atom contacts  |T|=33.58 A
  T=(-1, 0,-1)    13 atom-atom contacts  |T|=36.66 A
  T=(-1, 0, 0)    25 atom-atom contacts  |T|=27.42 A
  T=( 0,-1, 0)    46 atom-atom contacts  |T|=32.13 A
  T=( 0,-1, 1)     9 atom-atom contacts  |T|=46.60 A
  T=( 0, 0,-1)    47 atom-atom contacts  |T|=34.51 A

D(Q=0) eigenvalues: [    0.         0.         0.     50070.544  65476.3752 77591.994 ]

Exactly 3 of these should be ~0 (uniform rigid translation of the whole crystal -- the true acoustic zero modes). The other 3 are the zone-center "libration" (rotational) mode frequencies: these are physical and generally nonzero for an infinite periodic lattice.
